# Phase 1 Closure — Schema, Token Budget, and Duplication Audit

This notebook executes three of the six `docs/proposals/phase1_closure_prereg.md`
§9 pre-registration deliverables, against the **real** CORD v2 dataset and the
**real** pinned `Qwen/Qwen3-VL-4B-Instruct` processor:

1. **§9 deliverable 1** — generate `configs/cord_v2_output.schema.json` (the
   §6.5 normative output schema) from the real train+validation corpus.
2. **§9 deliverable 2** — measure the real §4 token distributions from the
   pinned processor and derive the frozen-formula token budget
   (`max_new_tokens`, `max_seq_len`, `eval_prefix_upper_bound`).
3. **§9 deliverable 5** — run the ADR-008/ADR-019/ADR-023 duplication audit
   (`src/vlm_lab/duplication.py`) on the real dataset and publish its
   aggregate `public_report()` output.

## Non-goals (read before proceeding)

This notebook is **Phase 1 closure work**. It does **not** start Phase 2.

- **No model inference or generation.** Only `processor.apply_chat_template`
  (tokenization/templating) runs — never `model.generate()`, and no model
  weights are ever loaded.
- **No training.**
- **No `notebooks/02_baseline.ipynb`, no Phase-2 code.**
- Per ADR-008 / `AGENTS.md` §15, this notebook **never displays, prints, or
  logs any test-split image or ground-truth content** — only aggregate
  counts and the duplication audit's `public_report()`. See the "Test-Split
  Blindness Policy" section below before running Part E.

`docs/proposals/phase1_closure_prereg.md` is a **draft proposal**, not yet
authoritative — this notebook produces the *measurements* §9 needs to close
it out; promoting the proposal's content into `EXPERIMENT_SPEC.md` /
`EVALUATION_PROTOCOL.md` / `IMPLEMENTATION_PLAN.md` / new ADRs (§9 item 9) is
a separate, later step.

## Environment setup

This notebook is meant to be runnable on its own, from a fresh Colab runtime,
via **Restart & Run All**. It reuses the exact repo-clone / install / staleness
self-heal / revision-guard pattern from `notebooks/01_dataset.ipynb`, adapted
to check the additional `vlm_lab` API this notebook depends on.

Unlike `00_environment.ipynb`, this notebook does **not** need a GPU: it only
loads the processor/tokenizer and config (no model weights), and only
performs CPU-side tokenization and templating. It should run identically on a
CPU-only Colab runtime or locally.

In [1]:
# Detect whether this notebook is running on Google Colab.
# This gates the Colab-only setup cell below, mirroring notebooks/01_dataset.ipynb.
import sys

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running on Google Colab: {IN_COLAB}")

Running on Google Colab: False


In [2]:
REPO_URL = "https://github.com/Mr-Kondo/finetuning_vlm.git"

# The split-scoped loader work (load_development_splits / mechanical_access /
# sealed_test) lives on an unmerged branch, so a plain clone of the default
# branch would check out a `src/vlm_lab/` that predates it. That mismatch is
# invisible until an `ImportError` fires several cells later, because the
# notebook and the code it imports come from different places: the notebook
# from wherever you opened it, the code from whatever this clone checks out.
# TODO: reset to "" once this branch is merged into main.
GIT_REF = "worktree-phase2-gate"  # "" = default branch

if IN_COLAB:
    import importlib
    import os
    import re
    import site
    import subprocess
    import sys
    import tomllib

    if not REPO_URL:
        raise RuntimeError(
            "IN_COLAB is True but REPO_URL is not set. Set REPO_URL above before "
            "running this notebook on Colab -- the very next cell depends on "
            "`vlm_lab` being installed."
        )

    repo_dir = "repo"

    if os.path.isdir(repo_dir):
        print(f"'{repo_dir}' already exists -- updating instead of re-cloning "
              "(idempotent across kernel restarts).")
        fetch = subprocess.run(
            ["git", "-C", repo_dir, "fetch", "origin"], capture_output=True, text=True
        )
        if fetch.returncode != 0:
            raise RuntimeError(
                f"git fetch failed (exit code {fetch.returncode}):\n{fetch.stderr}"
            )
        if GIT_REF:
            checkout = subprocess.run(
                ["git", "-C", repo_dir, "checkout", GIT_REF], capture_output=True, text=True
            )
            if checkout.returncode != 0:
                raise RuntimeError(
                    f"git checkout {GIT_REF!r} failed (exit code {checkout.returncode}):\n"
                    f"{checkout.stderr}"
                )
        pull = subprocess.run(
            ["git", "-C", repo_dir, "pull"], capture_output=True, text=True
        )
        if pull.returncode != 0:
            raise RuntimeError(
                f"git pull failed (exit code {pull.returncode}):\n{pull.stderr}"
            )
    else:
        clone_cmd = ["git", "clone"]
        if GIT_REF:
            clone_cmd += ["--branch", GIT_REF]
        clone_cmd += [REPO_URL, repo_dir]
        clone = subprocess.run(clone_cmd, capture_output=True, text=True)
        if clone.returncode != 0:
            raise RuntimeError(
                f"git clone failed (exit code {clone.returncode}):\n{clone.stderr}"
            )

    os.chdir(repo_dir)

    # Parse pyproject.toml's exact `==` pins directly, rather than duplicating
    # version strings in this notebook, so the two can never drift apart.
    with open("pyproject.toml", "rb") as f:
        _pyproject = tomllib.load(f)
    _pinned_versions: dict[str, str] = {}
    for _dep in _pyproject["project"]["dependencies"]:
        _match = re.match(r"^([A-Za-z0-9_.-]+)==([A-Za-z0-9_.+-]+)$", _dep.strip())
        if _match:
            _pinned_versions[_match.group(1).lower()] = _match.group(2)

    # Use `sys.executable -m pip`, not a bare `pip` command: Colab can have
    # more than one Python/pip on PATH, and a bare `pip` invocation can
    # install into a different interpreter's site-packages than the one
    # actually running this notebook.
    install = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", ".[dev]"],
        capture_output=True, text=True,
    )
    # Always show pip's actual resolution log, not just on failure -- this is
    # the only way to see whether pip genuinely reinstalled a pinned package
    # to the exact pinned version, versus deciding an already-installed
    # version was "close enough" for some transitive reason.
    print("--- pip install output ---")
    print(install.stdout)
    if install.stderr:
        print(install.stderr)
    print("--- end pip install output ---")
    if install.returncode != 0:
        raise RuntimeError(f"pip install failed (exit code {install.returncode}).")

    # `torchaudio` is not a dependency of this project (`transformers` only
    # declares it as an optional "audio"/"all"/"dev" extra; this project only
    # handles receipt images, never audio), so the `pip install` above never
    # touches it -- it is whatever Colab's base VM image happens to
    # pre-install, compiled against whichever CUDA version that image shipped
    # with. When the pinned `torch` build resolves to a different CUDA
    # version, importing it elsewhere can hard-fail with a "PyTorch and
    # TorchAudio were compiled with different CUDA versions" RuntimeError
    # (observed directly in notebooks/00_environment.ipynb's processor-load
    # cell). Removed here too for consistency.
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "torchaudio"],
        capture_output=True, text=True,
    )

    # Colab's own kernel startup imports PIL before any of this notebook's
    # cells run, and torch/torchvision may already be loaded too. Files just
    # reinstalled above on disk do NOT retroactively fix an already-imported
    # module in this same process -- especially for packages with compiled C
    # extensions, which cannot be safely hot-reloaded. Compare the
    # already-imported versions against the pins parsed above; if they don't
    # match, the files on disk are now correct but this process's cached
    # modules are not, so trigger a real interpreter restart and let the next
    # kernel start pick up the now-correct files.
    _version_check_targets = {"torch": "torch", "torchvision": "torchvision", "PIL": "pillow"}
    _mismatched = []
    for _module_name, _pin_key in _version_check_targets.items():
        if _module_name not in sys.modules:
            continue
        _installed = getattr(sys.modules[_module_name], "__version__", None)
        _pinned = _pinned_versions.get(_pin_key)
        if _installed is None or _pinned is None:
            continue
        if _installed.split("+")[0] != _pinned:
            _mismatched.append((_module_name, _installed, _pinned))

    if _mismatched:
        print(f"Already-imported package(s) do not match the pinned versions: {_mismatched}")
        print(
            "The correct versions are now installed on disk, but this running "
            "kernel already has the old ones cached in memory and cannot use "
            "the new files without an interpreter restart. Restarting the "
            "kernel now -- after it reconnects, run this notebook again "
            "(e.g. Restart & Run All); the git clone/pip install above will "
            "be fast no-ops since the files are already correct."
        )
        os.kill(os.getpid(), 9)

    # An editable install performed *after* this interpreter already started
    # writes a new `.pth` file into site-packages, but Python's `site` module
    # only processes `.pth` files at interpreter startup -- so `import vlm_lab`
    # still fails right after install with a plain `ModuleNotFoundError`.
    # `site.addsitedir()` re-processes each site-packages directory, including
    # the newly written `.pth` file, which fixes it.
    for site_dir in site.getsitepackages():
        site.addsitedir(site_dir)
    importlib.invalidate_caches()

    try:
        importlib.import_module("vlm_lab")
    except ImportError as exc:
        raise RuntimeError(
            "Repository was cloned and `pip install -e .[dev]` reported success, "
            f"but `import vlm_lab` still failed: {exc}"
        ) from exc

    # The clone above is the ONLY source of `vlm_lab`, and it can silently be a
    # different revision than the notebook expects (wrong branch, stale checkout
    # on a reused Colab disk). Verify the API this notebook actually needs is
    # present, and if not, say why rather than letting it surface later as a
    # bare ImportError.
    def _git(*args):
        """Run a read-only git command in the checkout, failing loudly."""
        result = subprocess.run(
            ["git", *args], capture_output=True, text=True
        )
        if result.returncode != 0 or not result.stdout.strip():
            raise RuntimeError(
                f"`git {' '.join(args)}` failed in {os.getcwd()!r} "
                f"(exit code {result.returncode}): {result.stderr.strip()!r}. "
                "The revision check below cannot run, so this cell refuses to "
                "report success rather than printing an unknown revision."
            )
        return result.stdout.strip()

    checked_out_branch = _git("rev-parse", "--abbrev-ref", "HEAD")
    checked_out_head = _git("rev-parse", "--short", "HEAD")
    print(f"Checked out {checked_out_branch} @ {checked_out_head}")

    import vlm_lab.data
    import vlm_lab.duplication
    import vlm_lab.mechanical_access
    import vlm_lab.schema

    # This notebook additionally depends on vlm_lab.schema, vlm_lab.duplication
    # and vlm_lab.mechanical_access, beyond notebooks/01_dataset.ipynb's
    # vlm_lab.data-only check.
    REQUIRED_DATA_API = {
        "vlm_lab.data": (vlm_lab.data, ("load_development_splits", "convert_ground_truth")),
        "vlm_lab.schema": (vlm_lab.schema, ("build_output_schema", "schema_hash", "validate_against_schema")),
        "vlm_lab.duplication": (vlm_lab.duplication, ("audit_duplication", "public_report", "ReceiptContent")),
        "vlm_lab.mechanical_access": (vlm_lab.mechanical_access, ("load_all_splits_for_mechanical_check",)),
    }
    missing = {
        module_name: [name for name in required_names if not hasattr(module, name)]
        for module_name, (module, required_names) in REQUIRED_DATA_API.items()
    }
    missing = {module_name: names for module_name, names in missing.items() if names}
    if missing:
        raise RuntimeError(
            f"Missing required API: {missing} -- the checked-out revision is not the "
            f"one this notebook was written against. GIT_REF is {GIT_REF!r} and HEAD is "
            f"{checked_out_branch!r} @ {checked_out_head!r}. Set GIT_REF to the "
            "branch that contains this API, or use "
            "Runtime -> Disconnect and delete runtime to clear a stale checkout. "
            "Do NOT 'fix' this by editing the import."
        )

    print("Repository cloned, package installed, and required vlm_lab API confirmed present.")
else:
    print("Not running on Colab -- skipping repo clone / install (package already installed locally).")

Not running on Colab -- skipping repo clone / install (package already installed locally).


## Test-Split Blindness Policy (read before proceeding)

Per `docs/DECISIONS.md` **ADR-008** and `docs/proposals/phase1_closure_prereg.md`
§8.1, this notebook's relationship with the held-out `test` split is:

- **Parts A, B, C, D** (schema generation, token-budget measurement, writing
  `configs/derived_budget.yaml`) use **`train` + `validation` only**, loaded
  through `vlm_lab.data.load_development_splits()` -- a function that has *no
  parameter* through which `test` could be named.
- **Part E** (the duplication audit) is the **only** place in this notebook
  that touches `test`, and it does so **mechanically only**: through the
  access-logged `vlm_lab.mechanical_access.load_all_splits_for_mechanical_check()`,
  to compute image hashes and ground-truth hashes for cross-split duplicate
  detection. No test-split image, ground-truth field, or annotation content is
  ever displayed, printed, or otherwise used to inform any decision.
- Part E prints **only** `vlm_lab.duplication.public_report()`'s aggregate
  counts and verdict -- never hashes, receipt IDs, or pair lists.
- Part E also writes a **sealed** per-receipt exclusion manifest to
  `results/sealed/duplication_audit_manifest.jsonl`. `.gitignore` already
  excludes `results/sealed/`, so that file will not be committed -- but this
  notebook must also never *print* its contents, since a printed cell output
  would still be committed if this notebook is saved via Colab's "Save a copy
  in GitHub". Only a line count is ever printed for that file.

## Pinned revisions and environment versions

Both the model and dataset revisions are pinned per ADR-015
(`docs/proposals/phase1_closure_prereg.md` §1), so every measurement below is
reproducible against exactly these commits. Library versions are recorded per
`AGENTS.md` §22.

In [3]:
import datasets
import torch
import transformers

MODEL_REPO = "Qwen/Qwen3-VL-4B-Instruct"
MODEL_REVISION = "ebb281ec70b05090aa6165b016eac8ec08e71b17"  # ADR-015

DATASET_REVISION = "7f0115a4b758a71d6473b8d085751692da2fef98"  # ADR-015
# vlm_lab.data.CORD_V2_REPO_ID is "naver-clova-ix/cord-v2"; not repeated as a
# literal here so the two can never drift apart.
from vlm_lab.data import CORD_V2_REPO_ID  # noqa: E402

print(f"Model:   {MODEL_REPO} @ {MODEL_REVISION}")
print(f"Dataset: {CORD_V2_REPO_ID} @ {DATASET_REVISION}")
print()
print(f"Python version:       {sys.version.split()[0]}")
print(f"transformers version: {transformers.__version__}")
print(f"torch version:        {torch.__version__}")
print(f"datasets version:     {datasets.__version__}")
print(f"CUDA available:       {torch.cuda.is_available()} "
      "(expected False/irrelevant -- this notebook is CPU-only by design; "
      "no model weights are loaded and no generation is performed)")

Model:   Qwen/Qwen3-VL-4B-Instruct @ ebb281ec70b05090aa6165b016eac8ec08e71b17
Dataset: naver-clova-ix/cord-v2 @ 7f0115a4b758a71d6473b8d085751692da2fef98

Python version:       3.12.14
transformers version: 5.15.0
torch version:        2.13.0
datasets version:     5.0.1
CUDA available:       False (expected False/irrelevant -- this notebook is CPU-only by design; no model weights are loaded and no generation is performed)


### Resolve the repository root

Every artifact this notebook writes (`configs/...`, `results/...`) is a
repository-root-relative path. In Colab the setup cell above already
`os.chdir`'d into the cloned repo, so this is a no-op there; run locally
(e.g. via `jupyter nbconvert --execute` from an arbitrary working directory),
`Path.cwd()` is not guaranteed to be the repo root, so it is located
explicitly rather than assumed.

In [4]:
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """Walk upward from `start` until a directory containing pyproject.toml is found."""
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError(
        f"Could not locate the repository root (a directory containing "
        f"pyproject.toml) starting from {start}. This notebook writes "
        "configs/ and results/ artifacts relative to the repository root "
        "and cannot proceed without it."
    )


REPO_ROOT = _find_repo_root(Path.cwd())
print(f"Repository root resolved to: {REPO_ROOT}")

Repository root resolved to: /Volumes/SSD/Programming/finetuning_vlm/.claude/worktrees/phase2-gate


## Load the pinned model processor (CPU only, no model weights)

Only `AutoConfig` and `AutoProcessor` are loaded -- never the ~4B-parameter
model itself. This is enough to run `apply_chat_template` for real tokenization
and image-grid measurement, and needs no GPU.

The processor's *shipped default* size caps are far larger than what §4
pre-registers (`longest_edge: 1_048_576`, `shortest_edge: 200_704` -- see
`docs/proposals/phase1_closure_prereg.md` §4 step 1). The shipped default is
printed below for the record, then overridden to the pre-registered caps
before any measurement is taken, since the token budget must be derived under
the caps this experiment will actually run under, not the library's defaults.

In [5]:
from transformers import AutoConfig, AutoProcessor

# §4 step 1 -- the pre-registered processor size caps, NOT the processor's
# shipped defaults.
PROCESSOR_SIZE = {"longest_edge": 1_048_576, "shortest_edge": 200_704}

print(f"Loading AutoConfig for {MODEL_REPO}@{MODEL_REVISION} (config only, no weights)...")
model_config = AutoConfig.from_pretrained(MODEL_REPO, revision=MODEL_REVISION)

print(f"Loading AutoProcessor for {MODEL_REPO}@{MODEL_REVISION} (CPU; no GPU required)...")
processor = AutoProcessor.from_pretrained(MODEL_REPO, revision=MODEL_REVISION)

shipped_default_size = dict(processor.image_processor.size)
print(f"Processor's shipped default size caps: {shipped_default_size}")
processor.image_processor.size = dict(PROCESSOR_SIZE)
print(f"Overridden to the §4 pre-registered caps:  {processor.image_processor.size}")

print()
print(f"Processor class:       {type(processor).__name__}")
print(f"Tokenizer class:       {type(processor.tokenizer).__name__}")
print(f"Image processor class: {type(processor.image_processor).__name__}")

Loading AutoConfig for Qwen/Qwen3-VL-4B-Instruct@ebb281ec70b05090aa6165b016eac8ec08e71b17 (config only, no weights)...
Loading AutoProcessor for Qwen/Qwen3-VL-4B-Instruct@ebb281ec70b05090aa6165b016eac8ec08e71b17 (CPU; no GPU required)...


[ERROR] `min_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /private/tmp/vlm_lab_venv/lib/python3.12/site-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.
[ERROR] `max_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /private/tmp/vlm_lab_venv/lib/python3.12/site-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.
Processor's shipped default size caps: {'longest_edge': 16777216, 'shortest_edge': 65536}
Overridden to the §4 pre-registered caps:  {'longest_edge': 1048576, 'shortest_edge': 200704}

Processor class:       Qwen3VLProcessor
Tokenizer class:       Qwen2Tokenizer
Image processor class: Qwen2VLImageProcessor


In [6]:
# §4/§3.1 assume patch_size=16, spatial_merge_size=2. Verified against the
# real pinned processor/model config here rather than hardcoded, per the
# task's explicit instruction not to hardcode these blindly.
PATCH_SIZE = processor.image_processor.patch_size
SPATIAL_MERGE_SIZE = processor.image_processor.merge_size
EOS_TOKEN_ID = processor.tokenizer.eos_token_id

# The pinned chat template renders '<|im_end|>\n' after every turn, including
# the assistant's -- and that string does NOT collapse to a single token: the
# newline is a separate, constant, content-independent token. Derived here
# rather than hardcoded, so a future tokenizer revision that changed this
# would fail this assertion instead of silently invalidating the section 5.4
# step-5 check below.
_newline_ids = processor.tokenizer.encode("\n", add_special_tokens=False)
assert len(_newline_ids) == 1, (
    f"expected the tokenizer to encode a bare newline as exactly one token, got {_newline_ids} "
    "-- the step-5 assertion below assumes this and needs re-deriving if it doesn't hold."
)
TEMPLATE_NEWLINE_TOKEN_ID = _newline_ids[0]

IMAGE_TOKEN_ID = getattr(model_config, "image_token_id", None)
MAX_POSITION_EMBEDDINGS = getattr(
    getattr(model_config, "text_config", model_config), "max_position_embeddings", None
)

print(f"patch_size (real, processor.image_processor.patch_size):         {PATCH_SIZE}")
print(f"spatial_merge_size (real, processor.image_processor.merge_size): {SPATIAL_MERGE_SIZE}")
print(f"eos_token_id (real, processor.tokenizer.eos_token_id):           {EOS_TOKEN_ID}")
print(f"template_newline_token_id (real, encode of a bare newline):      {TEMPLATE_NEWLINE_TOKEN_ID}")
print(f"image_token_id (real, model_config.image_token_id):              {IMAGE_TOKEN_ID}")
print(f"max_position_embeddings (real, model_config.text_config):        {MAX_POSITION_EMBEDDINGS}")

_assumed_patch_size, _assumed_merge_size = 16, 2  # section 3.1/4's assumed values
if (PATCH_SIZE, SPATIAL_MERGE_SIZE) == (_assumed_patch_size, _assumed_merge_size):
    print(
        f"\nConfirmed: real (patch_size, spatial_merge_size) match the "
        f"proposal's assumed ({_assumed_patch_size}, {_assumed_merge_size})."
    )
else:
    print(
        f"\nNOTE: real (patch_size, spatial_merge_size) = "
        f"({PATCH_SIZE}, {SPATIAL_MERGE_SIZE}) differs from the proposal's "
        f"assumed ({_assumed_patch_size}, {_assumed_merge_size}); using the "
        "REAL values for every derived formula below."
    )

assert MAX_POSITION_EMBEDDINGS is not None, (
    "Could not find max_position_embeddings on the pinned model config "
    "(checked model_config.text_config and model_config itself) -- the section 4 "
    "max_seq_len assertion below has nothing to check against."
)


patch_size (real, processor.image_processor.patch_size):         16
spatial_merge_size (real, processor.image_processor.merge_size): 2
eos_token_id (real, processor.tokenizer.eos_token_id):           151645
template_newline_token_id (real, encode of a bare newline):      198
image_token_id (real, model_config.image_token_id):              151655
max_position_embeddings (real, model_config.text_config):        262144

Confirmed: real (patch_size, spatial_merge_size) match the proposal's assumed (16, 2).


## Part A — Load the real data (train + validation)

Parts A–D use **`train` + `validation` only**. All three splits are touched
only in Part E, through a separate access-logged loader (see the Test-Split
Blindness Policy above).

In [7]:
from vlm_lab.data import convert_ground_truth, load_development_splits

# Real network call to the Hugging Face Hub -- not mocked.
# `load_development_splits` cannot name the held-out `test` split: there is no
# parameter through which a caller could ask for it.
dataset = load_development_splits(revision=DATASET_REVISION)
print("load_development_splits() succeeded.")

load_development_splits() succeeded.


In [8]:
EXPECTED_DEVELOPMENT_SPLIT_SIZES = {"train": 800, "validation": 100}

print("Development split sizes:")
all_match = True
for split_name, expected_count in EXPECTED_DEVELOPMENT_SPLIT_SIZES.items():
    actual_count = len(dataset[split_name])
    status = "OK" if actual_count == expected_count else "MISMATCH"
    if actual_count != expected_count:
        all_match = False
    print(f"  {split_name:>10}: {actual_count} rows (expected {expected_count}) [{status}]")

assert all_match, "Split sizes do not match docs/EXPERIMENT_SPEC.md §3 / ADR-007 expectations."
print("\nPASS: train=800, validation=100 as expected.")

Development split sizes:
       train: 800 rows (expected 800) [OK]
  validation: 100 rows (expected 100) [OK]

PASS: train=800, validation=100 as expected.


## Part B — Schema generation (§6.5, §9 deliverable 1)

`vlm_lab.schema.build_output_schema()` implements the §6.5 deterministic
construction algorithm. It is run here over the real train+validation corpus.

**MAJOR FINDING, not silently patched around.** The real corpus contains a
widespread, previously-undiscovered pattern the frozen §6.5 algorithm cannot
encode: **at least 14 distinct leaf paths** (`menu[].cnt`, `menu[].nm`,
`menu[].discountprice`, `menu[].num`, `sub_total.etc`,
`sub_total.subtotal_price`, `sub_total.tax_price`,
`sub_total.discount_price`, `sub_total.othersvc_price`, `sub_total` itself,
`total.cashprice`, `total.changeprice`, `total.creditcardprice`,
`total.total_price`) have a value that is a **list of strings** on a small
but non-trivial share of records (57 of 900, ≈6.3%), where the same field is
a plain string everywhere else. Two distinct sub-patterns were observed on
inspection (train/validation only, always safe to display):

- **Exact-duplicate collapse** — e.g. `subtotal_price: ["20,000", "20,000"]`,
  `cashprice: ["100,000", "100,000"]` — the same value repeated.
- **Genuinely distinct values** — e.g.
  `etc: ["1,213,000", "60,000", "70,000"]`,
  `cashprice: ["74,000", "100,000"]` — different amounts under one key.

This is almost certainly the **same class of Donut serialization quirk**
`convert_ground_truth` already normalizes for the `menu`/`void_menu`/`sub`
**group** keys (a value that collapses to a bare form when Donut's
annotation pipeline detects it once but should be a list) — except here it
is happening on **scalar leaf fields**, which `convert_ground_truth` has
never normalized, because nothing before this schema-generation work has
ever walked every leaf of the full corpus exhaustively enough to notice.

**This has a consequence beyond schema generation.** `convert_ground_truth`
also produces **training targets** (§5.4). For these same ~57 records, the
JSON the model would be trained to reproduce currently contains a list where
the §5.3 prompt promises a plain string — a genuine, pre-existing data-shape
inconsistency in the training signal for a real 6% slice of the corpus, not
merely a schema-generation inconvenience.

**What this notebook does, and does not do, about it:** it detects every
offending path and the exact record(s) responsible, reports them fully
below, builds the schema **excluding only those specific records**, and then
validates the **full** 900-record corpus against that schema so every
exclusion's consequence stays visible as a reported round-trip failure. It
does **not** decide how `convert_ground_truth` should ultimately normalize
these fields — the two sub-patterns above (exact duplicate vs. genuinely
distinct values) plausibly need different treatment, and different fields
may need different rules; that is a real specification decision, escalated
in `docs/proposals/phase1_closure_prereg.md` and `docs/STATE.md`, not made
here.


In [9]:
train_and_validation_ground_truths = [
    convert_ground_truth(row["ground_truth"]) for row in dataset["train"]
] + [
    convert_ground_truth(row["ground_truth"]) for row in dataset["validation"]
]
print(f"Corpus size: {len(train_and_validation_ground_truths)} converted ground truths "
      f"({len(dataset['train'])} train + {len(dataset['validation'])} validation).")

Corpus size: 900 converted ground truths (800 train + 100 validation).


In [10]:
import json
from collections import Counter

import vlm_lab.schema


def _value_at_path(document, path_after_root):
    """Walk `document` along a dotted/`[]` path (e.g. 'menu[].cnt') and yield
    every leaf value reached, treating a `[]`-suffixed segment as "for each
    element of this list"."""
    if not path_after_root:
        yield document
        return
    segment, _, rest = path_after_root.partition(".")
    if segment.endswith("[]"):
        key = segment[:-2]
        if key:
            if not isinstance(document, dict) or key not in document:
                return
            document = document[key]
        if not isinstance(document, list):
            return
        for element in document:
            yield from _value_at_path(element, rest)
    else:
        if not isinstance(document, dict) or segment not in document:
            return
        yield from _value_at_path(document[segment], rest)


def _parse_offending_path(message):
    """Extract the quoted path from a §6.5 SchemaShapeError message."""
    path_start = message.index("path ") + len("path ")
    quote = message[path_start]
    if quote not in ("'", chr(34)):
        return None
    path_end = message.index(quote, path_start + 1)
    return message[path_start + 1 : path_end]


def _resolve_anomaly(corpus, included, path_after_root):
    """Identify exactly which included corpus indices are responsible for a
    SchemaShapeError at `path_after_root`, using the SAME two rules the §6.5
    algorithm itself applies (not a "try removing one at a time and hope"
    heuristic, which breaks whenever two or more records share the anomaly
    at once -- as several of these paths do):

    - if this is an ARRAY-ELEMENT-shape error (path ends in "[]"), every
      candidate whose elements there include a non-object is a culprit --
      there is no "majority wins" here, since the algorithm's rule for array
      elements is fixed (objects only), not a popularity contest;
    - otherwise, the candidates disagreeing with the majority VALUE shape at
      this path are the culprits;
    - if every candidate agrees on one shape and it is STILL rejected (e.g.
      a field that is *always* an array of strings, never a plain string,
      so there is no majority "string" to fall back to), every candidate
      touching the path is the culprit -- the field cannot be encoded at all
      given the current corpus content.
    """
    if path_after_root.endswith("[]"):
        bad = [
            i for i in included
            if (elements := list(_value_at_path(corpus[i], path_after_root)))
            and any(vlm_lab.schema._shape_name(e) != "object" for e in elements)
        ]
        if bad:
            return bad

    shapes_by_candidate = {
        i: shapes
        for i in included
        if (shapes := {
            vlm_lab.schema._shape_name(v) for v in _value_at_path(corpus[i], path_after_root)
        })
    }
    if not shapes_by_candidate:
        return []
    shape_counts = Counter(shape for shapes in shapes_by_candidate.values() for shape in shapes)
    majority_shape, _ = shape_counts.most_common(1)[0]
    bad = [i for i, shapes in shapes_by_candidate.items() if shapes != {majority_shape}]
    return bad if bad else list(shapes_by_candidate)


def build_schema_reporting_anomalies(corpus, max_rounds=30):
    """Build the §6.5 schema, excluding only the specific corpus record(s)
    responsible for each SchemaShapeError -- reported, never silently
    absorbed. Returns (schema, excluded_record_indices, anomaly_reports)."""
    included = list(range(len(corpus)))
    excluded_indices = []
    anomaly_reports = []

    for _ in range(max_rounds):
        try:
            schema = vlm_lab.schema.build_output_schema([corpus[i] for i in included])
            return schema, excluded_indices, anomaly_reports
        except vlm_lab.schema.SchemaShapeError as exc:
            offending_path = _parse_offending_path(str(exc))
            if offending_path is None:
                raise
            path_after_root = (
                offending_path[2:] if offending_path.startswith("$.") else offending_path.lstrip("$")
            )
            culprits = _resolve_anomaly(corpus, included, path_after_root)
            if not culprits:
                raise RuntimeError(
                    f"SchemaShapeError at path {offending_path!r} but no culprit could be "
                    "identified -- cannot make progress."
                ) from exc
            included = [i for i in included if i not in culprits]
            excluded_indices.extend(culprits)
            anomaly_reports.append({
                "offending_path": offending_path,
                "excluded_corpus_indices": sorted(culprits),
            })
    raise RuntimeError(f"Could not build a schema after {max_rounds} exclusion rounds.")


schema, excluded_schema_indices, anomaly_reports = build_schema_reporting_anomalies(
    train_and_validation_ground_truths
)

print(f"FINDING: {len(excluded_schema_indices)} corpus record(s) excluded from schema "
      f"construction across {len(anomaly_reports)} distinct anomalous path(s). This is a "
      "real, previously-undiscovered characteristic of the source CORD v2 data as processed "
      "by convert_ground_truth, not an implementation bug -- see the markdown cell above and "
      "docs/proposals/phase1_closure_prereg.md for the required disposition.")
for report in anomaly_reports:
    indices = report["excluded_corpus_indices"]
    print(f"\n  path: {report['offending_path']}  (excluded {len(indices)}: {indices})")
    for index in indices:
        split_name = "train" if index < len(dataset["train"]) else "validation"
        row_in_split = index if split_name == "train" else index - len(dataset["train"])
        print(f"    -> corpus index {index} = {split_name} row {row_in_split}: "
              f"{json.dumps(train_and_validation_ground_truths[index], ensure_ascii=False)}")

print(f"\nSchema built from {len(train_and_validation_ground_truths) - len(excluded_schema_indices)} "
      f"of {len(train_and_validation_ground_truths)} corpus records.")
print(f"Top-level keys: {sorted(schema['properties'].keys())}")


FINDING: 57 corpus record(s) excluded from schema construction across 14 distinct anomalous path(s). This is a real, previously-undiscovered characteristic of the source CORD v2 data as processed by convert_ground_truth, not an implementation bug -- see the markdown cell above and docs/proposals/phase1_closure_prereg.md for the required disposition.

  path: $.menu[].cnt  (excluded 1: [525])
    -> corpus index 525 = train row 525: {"menu": [{"unitprice": "17,000", "cnt": ["TRIPPLE CHEESE", "1"], "price": "17,000"}], "sub_total": {"subtotal_price": "17,000"}, "total": {"total_price": "17,000", "changeprice": "0", "emoneyprice": "17,000"}}

  path: $.menu[].discountprice  (excluded 3: [149, 235, 479])
    -> corpus index 149 = train row 149: {"menu": [{"nm": "#PKTPOLSBTSPON2S", "unitprice": "8,000", "cnt": "1X", "discountprice": "800-", "price": "8,000"}, {"nm": "BENECOL LYCHEE 2S", "unitprice": "14,000", "cnt": "1X", "discountprice": ["1,400 -", "1,260-"], "price": "14,000"}, {"nm": "R

In [11]:
import json

configs_dir = REPO_ROOT / "configs"
configs_dir.mkdir(parents=True, exist_ok=True)
schema_path = configs_dir / "cord_v2_output.schema.json"
with schema_path.open("w", encoding="utf-8") as f:
    json.dump(schema, f, indent=2, ensure_ascii=False)
print(f"Wrote {schema_path}")

Wrote /Volumes/SSD/Programming/finetuning_vlm/.claude/worktrees/phase2-gate/configs/cord_v2_output.schema.json


In [12]:
schema_hash = vlm_lab.schema.schema_hash(schema)
print(f"schema_hash = {schema_hash}")

schema_hash = 2c69ae8ab48b2d9b7558adba5cdde35780ab6c1034062890c10dc571fba741f9


In [13]:
# Round-trip check, run over the FULL corpus regardless of any schema-building
# exclusions above -- so an excluded record's disagreement with the schema
# stays visible as a reported round-trip failure rather than disappearing.
violations_by_index = {
    index: vlm_lab.schema.validate_against_schema(instance, schema)
    for index, instance in enumerate(train_and_validation_ground_truths)
}
failing = {index: v for index, v in violations_by_index.items() if v}

expected_failures = set(excluded_schema_indices)
unexpected_failures = {i: v for i, v in failing.items() if i not in expected_failures}
expected_but_passing = expected_failures - set(failing.keys())

print(f"{len(train_and_validation_ground_truths) - len(failing)} / "
      f"{len(train_and_validation_ground_truths)} corpus elements validate against the "
      "generated schema.")
if failing:
    print(f"\n{len(failing)} do not validate ({len(failing) - len(unexpected_failures)} "
          f"expected from the anomalies reported above, {len(unexpected_failures)} unexpected):")
    for index in sorted(failing)[:15]:
        expected_tag = " (expected)" if index in expected_failures else " (UNEXPECTED)"
        print(f"  index {index}{expected_tag}: {failing[index][:2]}"
              f"{' ...' if len(failing[index]) > 2 else ''}")

assert not unexpected_failures, (
    f"{len(unexpected_failures)} corpus element(s) failed round-trip validation that were NOT "
    f"already known/excluded above -- a genuinely new anomaly. Indices: {sorted(unexpected_failures)}"
)
assert not expected_but_passing, (
    f"{len(expected_but_passing)} corpus element(s) were excluded from schema construction but "
    f"unexpectedly DO validate against the resulting schema: {sorted(expected_but_passing)} -- "
    "the exclusion logic may be over-broad."
)
print(f"\nPASS: every round-trip failure is exactly one of the {len(expected_failures)} "
      "already-reported anomalies above, and nothing else -- the round-trip property holds for "
      f"the rest of the corpus ({len(train_and_validation_ground_truths) - len(expected_failures)} "
      "records).")


843 / 900 corpus elements validate against the generated schema.

57 do not validate (57 expected from the anomalies reported above, 0 unexpected):
  index 19 (expected): ["$.sub_total.etc: ['1,213,000', '60,000', '70,000'] is not of type 'string'"]
  index 56 (expected): ["$.sub_total.subtotal_price: ['49.636', '49.636'] is not of type 'string'"]
  index 88 (expected): ["$.sub_total.discount_price: ['0', '0'] is not of type 'string'"]
  index 103 (expected): ["$.sub_total.etc: ['797,000', '83,000', '60,000'] is not of type 'string'"]
  index 124 (expected): ["$.sub_total.etc: ['757,000', '100,000', '80,000'] is not of type 'string'"]
  index 135 (expected): ["$.total.total_price: ['55,834', '55,800'] is not of type 'string'"]
  index 149 (expected): ["$.menu[1].discountprice: ['1,400 -', '1,260-'] is not of type 'string'", "$.menu[5].discountprice: ['1,400 -', '790-'] is not of type 'string'"]
  index 150 (expected): ["$.sub_total.subtotal_price: ['20,000', '20,000'] is not of type 's

### Schema generation report

Total keys observed, maximum nesting depth, and any path whose only observed
array content was empty (§6.5 step 7 — such a path is *less* strict than the
rest of the schema, since `minItems` is deliberately not set there).

In [14]:
def _walk_schema(node, path):
    """Yield (path, node) for every object/array node in the generated schema."""
    yield path, node
    if node.get("type") == "object":
        for key, child in node.get("properties", {}).items():
            yield from _walk_schema(child, f"{path}.{key}")
    elif node.get("type") == "array" and "items" in node:
        yield from _walk_schema(node["items"], f"{path}[]")


all_nodes = list(_walk_schema(schema, "$"))
total_keys_observed = sum(
    len(node.get("properties", {})) for _, node in all_nodes if node.get("type") == "object"
)
max_nesting_depth = max(path.count(".") + path.count("[]") for path, _ in all_nodes)
empty_array_only_paths = [
    path for path, node in all_nodes
    if node.get("type") == "array" and "items" not in node
]

print(f"Total object-node keys observed across the schema: {total_keys_observed}")
print(f"Maximum nesting depth (dot/[] segments from root): {max_nesting_depth}")
if empty_array_only_paths:
    print(f"Array-only-ever-empty paths ({len(empty_array_only_paths)}), looser than the "
          "rest of the schema (no `items` constraint):")
    for path in empty_array_only_paths:
        print(f"  {path}")
else:
    print("No array-only-ever-empty paths: every array path observed at least one element.")

Total object-node keys observed across the schema: 31
Maximum nesting depth (dot/[] segments from root): 5
No array-only-ever-empty paths: every array path observed at least one element.


## Part C — Token budget measurement (§4, §9 deliverable 2)

§5.4's input-construction algorithm is implemented **inline in this notebook**
(it is a one-time measurement, not yet part of `src/vlm_lab` — that is
later-phase work). For every train+validation sample this measures:

1. `eval_prefix_len` — tokens in the standalone evaluation prefix
   (system + user, `add_generation_prompt=True`).
2. `train_seq_len` — tokens in the full training sequence
   (system + user + assistant target, `add_generation_prompt=False`).
3. `assistant_label_n` — count of non-`-100` label positions after
   assistant-only masking (§3.5, §5.4 step 3).
4. `image_token_n` — from the processor's real `image_grid_thw`, divided by
   `spatial_merge_size**2` — never from pixel-area arithmetic (`[F18]`/`[R3-3]`).

The frozen prompt text below (§5.3) is extracted **programmatically** from
`docs/proposals/phase1_closure_prereg.md`'s fenced code blocks at notebook-
authoring time and pasted here as a literal constant, to eliminate manual
transcription risk (verified byte-for-byte against the source document).

In [15]:
SYSTEM_PROMPT_TEXT = (
    "You are an information extraction model. You read a receipt image and return the extracted\n"
    "information as a single JSON object. You return only JSON. You never return explanations,\n"
    "commentary, or markdown code fences."
)

USER_PROMPT_TEXT = (
    "Extract the structured information from this receipt image as a single JSON object.\n"
    "\n"
    "Use exactly these top-level keys when the corresponding information is present, and omit a key\n"
    "entirely when it is not present:\n"
    "- \"menu\": a list of ordered items. Each item may contain \"nm\" (name), \"cnt\" (count),\n"
    "  \"price\", \"unitprice\", \"discountprice\", and \"sub\" (a list of sub-items with the same fields).\n"
    "- \"sub_total\": may contain \"subtotal_price\", \"discount_price\", \"service_price\", \"tax_price\",\n"
    "  \"etc\".\n"
    "- \"total\": may contain \"total_price\", \"cashprice\", \"changeprice\", \"creditcardprice\",\n"
    "  \"menuqty_cnt\", \"menutype_cnt\".\n"
    "- \"void_menu\": voided items, same structure as \"menu\".\n"
    "\n"
    "Transcribe every value exactly as printed on the receipt. Preserve digit grouping such as\n"
    "\"58,000\" or \"23.000\", currency symbols, capitalization, and internal spacing exactly as shown.\n"
    "Do not convert, round, reformat, translate, or tidy any value. Leading and trailing spaces around\n"
    "a value do not matter. Do not invent fields that are not visible on the receipt.\n"
    "\n"
    "Return only the JSON object."
)

print("SYSTEM_PROMPT_TEXT and USER_PROMPT_TEXT loaded (frozen §5.3 text).")

SYSTEM_PROMPT_TEXT and USER_PROMPT_TEXT loaded (frozen §5.3 text).


### §5.4 steps 4–5: the two per-sample assertions

Both are **hard** assertions — a failure stops the notebook immediately
rather than being averaged over, since either would mean training and
evaluation silently see different tokenizations of the "same" input, which is
exactly what ADR-006/§5.4 exist to prevent.

- **Step 4 (prefix-equality):** `full_features`'s first `prefix_len` tokens
  must exactly equal `prefix_features`.
- **Step 5 (terminal pair), as corrected below §5.4 in
  `docs/proposals/phase1_closure_prereg.md`:** the pinned chat template
  renders `'<|im_end|>\n'` after every turn, including the assistant's, and
  that string tokenizes as **two** constant tokens, not one — confirmed
  directly against the real pinned `tokenizer.json` before this notebook was
  finalized (`tokenizer.encode("<|im_end|>\n")` -> `[151645, 198]` in every
  case tried, empty JSON object included). So the assertion is
  `input_ids[-2] == eos_token_id` and `input_ids[-1] == TEMPLATE_NEWLINE_TOKEN_ID`
  — not `input_ids[-1] == eos_token_id`, which the pre-registration said
  through v3.2 and which fails against the real template. The loop below also
  confirms the assistant turn contributes **exactly one** new `eos_token_id`
  beyond the prefix (no duplicated terminators).


In [16]:
import math

train_and_validation_samples = [
    (row["image"], row["ground_truth"]) for row in dataset["train"]
] + [
    (row["image"], row["ground_truth"]) for row in dataset["validation"]
]
print(f"Measuring over {len(train_and_validation_samples)} train+validation samples.")

eval_prefix_lens = []
train_seq_lens = []
assistant_label_ns = []
image_token_ns = []
fixed_prompt_and_template_token_values = set()

for sample_index, (image, ground_truth_json) in enumerate(train_and_validation_samples):
    converted_ground_truth = convert_ground_truth(ground_truth_json)

    messages_eval = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT_TEXT}]},
        {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": USER_PROMPT_TEXT}]},
    ]
    prefix_features = processor.apply_chat_template(
        messages_eval, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    )
    prefix_len = prefix_features["input_ids"].shape[-1]
    eval_prefix_lens.append(prefix_len)

    assistant_text = json.dumps(
        converted_ground_truth, ensure_ascii=False, separators=(",", ":"), sort_keys=False
    )
    messages_full = messages_eval + [
        {"role": "assistant", "content": [{"type": "text", "text": assistant_text}]}
    ]
    full_features = processor.apply_chat_template(
        messages_full, tokenize=True, add_generation_prompt=False,
        return_dict=True, return_tensors="pt",
    )
    full_len = full_features["input_ids"].shape[-1]
    train_seq_lens.append(full_len)

    # §5.4 step 3: assistant-only label masking. No manual EOS append (§5.4
    # step 2 / [R3-2]) -- the template already emits the terminator pair.
    labels = full_features["input_ids"].clone()
    labels[:, :prefix_len] = -100
    assistant_label_n = int((labels != -100).sum().item())
    assistant_label_ns.append(assistant_label_n)

    # §5.4 step 4 -- HARD assertion.
    prefix_matches = full_features["input_ids"][0, :prefix_len].equal(prefix_features["input_ids"][0])
    assert prefix_matches, (
        f"§5.4 step 4 FAILED at train+validation sample {sample_index}: the full training "
        "sequence's prefix does not exactly match the standalone evaluation prefix. Stopping "
        "immediately rather than averaging over it -- see the markdown note above this cell."
    )

    # §5.4 step 5 -- HARD assertion, corrected: EOS is the second-to-last
    # token, the fixed template newline is last (see the markdown note above
    # this cell for why input_ids[-1] == eos_token_id is wrong here).
    full_ids = full_features["input_ids"][0].tolist()
    terminal_pair_correct = (
        full_ids[-2] == EOS_TOKEN_ID and full_ids[-1] == TEMPLATE_NEWLINE_TOKEN_ID
    )
    assert terminal_pair_correct, (
        f"§5.4 step 5 FAILED at train+validation sample {sample_index}: expected the terminal "
        f"pair (eos_token_id={EOS_TOKEN_ID}, template_newline_token_id={TEMPLATE_NEWLINE_TOKEN_ID}) "
        f"at positions [-2:], got {full_ids[-2:]}. Stopping immediately -- see the markdown note "
        "above this cell."
    )

    # Substantive corroboration: beyond the prefix, the assistant turn
    # contributes exactly one NEW eos_token_id occurrence (no duplicated
    # terminators such as the v2-era manual-EOS-append bug, [R3-2]).
    new_eos_count = full_ids[prefix_len:].count(EOS_TOKEN_ID)
    assert new_eos_count == 1, (
        f"§5.4 step 5 FAILED at train+validation sample {sample_index}: the assistant turn "
        f"contributed {new_eos_count} new eos_token_id occurrence(s) beyond the prefix, expected "
        "exactly 1. Stopping immediately -- see the markdown note above this cell."
    )

    # image_token_n from the REAL processor grid, never pixel-area arithmetic.
    image_grid_thw = full_features["image_grid_thw"][0]
    image_token_n = int(image_grid_thw.prod().item()) // (SPATIAL_MERGE_SIZE ** 2)
    image_token_ns.append(image_token_n)

    fixed_prompt_and_template_token_values.add(prefix_len - image_token_n)

print(
    f"Processed {len(train_and_validation_samples)} samples. §5.4 steps 4 and 5 (corrected: "
    "EOS at position -2, template newline at -1) held for all of them -- the loop would have "
    "stopped otherwise."
)


Measuring over 900 train+validation samples.


Processed 900 samples. §5.4 steps 4 and 5 (corrected: EOS at position -2, template newline at -1) held for all of them -- the loop would have stopped otherwise.


In [17]:
if len(fixed_prompt_and_template_token_values) == 1:
    fixed_prompt_and_template_tokens = next(iter(fixed_prompt_and_template_token_values))
    print(f"fixed_prompt_and_template_tokens is CONSTANT across all samples: "
          f"{fixed_prompt_and_template_tokens}")
else:
    print(
        f"FINDING: fixed_prompt_and_template_tokens is NOT constant -- observed "
        f"{len(fixed_prompt_and_template_token_values)} distinct values: "
        f"{sorted(fixed_prompt_and_template_token_values)}. Reporting the full distribution "
        "rather than silently picking one value, per the pre-registration's own instruction."
    )
    fixed_prompt_and_template_tokens = max(fixed_prompt_and_template_token_values)
    print(f"Using the maximum observed value ({fixed_prompt_and_template_tokens}) conservatively "
          "for the §4 formulas below, since the budget must not underestimate the prompt's real "
          "token cost.")


fixed_prompt_and_template_tokens is CONSTANT across all samples: 321


### Aggregate distributions (descriptive reporting only, per §4 — these are
not budget inputs)

In [18]:
def _percentile(values, percentile):
    """Nearest-rank percentile over `values` (0 <= percentile <= 100)."""
    ordered = sorted(values)
    if not ordered:
        raise ValueError("cannot compute a percentile of an empty sequence")
    rank = max(0, min(len(ordered) - 1, math.ceil(percentile / 100 * len(ordered)) - 1))
    return ordered[rank]


def _summarize(name, values):
    summary = {"p50": _percentile(values, 50), "p95": _percentile(values, 95), "max": max(values)}
    print(f"{name:>20}: p50={summary['p50']:>5}  p95={summary['p95']:>5}  "
          f"max={summary['max']:>5}  (n={len(values)})")
    return summary


measured_distributions = {
    "eval_prefix_len": _summarize("eval_prefix_len", eval_prefix_lens),
    "train_seq_len": _summarize("train_seq_len", train_seq_lens),
    "assistant_label_n": _summarize("assistant_label_n", assistant_label_ns),
    "image_token_n": _summarize("image_token_n", image_token_ns),
}

     eval_prefix_len: p50= 1329  p95= 1335  max= 1345  (n=900)
       train_seq_len: p50= 1409  p95= 1557  max= 1867  (n=900)
   assistant_label_n: p50=  102  p95=  250  max=  575  (n=900)
       image_token_n: p50= 1008  p95= 1014  max= 1024  (n=900)


### Frozen §4 formulas

Using the measured **constant** `fixed_prompt_and_template_tokens`, not any
observed maximum (`[R3-3]`'s whole point: a held-out test image can produce a
larger grid while still obeying `processor_size`, so the budget must be sized
from the *allowed* ceiling, not the *observed* maximum).

In [19]:
IMAGE_TOKEN_CEILING = PROCESSOR_SIZE["longest_edge"] // (PATCH_SIZE * SPATIAL_MERGE_SIZE) ** 2
EVAL_PREFIX_UPPER_BOUND = IMAGE_TOKEN_CEILING + fixed_prompt_and_template_tokens
MAX_NEW_TOKENS = math.ceil(max(assistant_label_ns) * 1.25)
MAX_SEQ_LEN = max(max(train_seq_lens), EVAL_PREFIX_UPPER_BOUND + MAX_NEW_TOKENS)

print(f"image_token_ceiling     = {PROCESSOR_SIZE['longest_edge']} // "
      f"({PATCH_SIZE} * {SPATIAL_MERGE_SIZE})**2 = {IMAGE_TOKEN_CEILING}")
print(f"eval_prefix_upper_bound = {IMAGE_TOKEN_CEILING} + {fixed_prompt_and_template_tokens} "
      f"= {EVAL_PREFIX_UPPER_BOUND}")
print(f"max_new_tokens          = ceil({max(assistant_label_ns)} * 1.25) = {MAX_NEW_TOKENS}")
print(f"max_seq_len             = max({max(train_seq_lens)}, {EVAL_PREFIX_UPPER_BOUND} + "
      f"{MAX_NEW_TOKENS}) = {MAX_SEQ_LEN}")

assert MAX_SEQ_LEN <= MAX_POSITION_EMBEDDINGS, (
    f"max_seq_len ({MAX_SEQ_LEN}) exceeds the pinned model's max_position_embeddings "
    f"({MAX_POSITION_EMBEDDINGS}) -- the §4 gate fails loudly rather than silently truncating."
)
print(f"\nPASS: max_seq_len ({MAX_SEQ_LEN}) <= max_position_embeddings ({MAX_POSITION_EMBEDDINGS}).")

image_token_ceiling     = 1048576 // (16 * 2)**2 = 1024
eval_prefix_upper_bound = 1024 + 321 = 1345
max_new_tokens          = ceil(575 * 1.25) = 719
max_seq_len             = max(1867, 1345 + 719) = 2064

PASS: max_seq_len (2064) <= max_position_embeddings (262144).


## Part D — Write the shared artifact contract for `notebooks/01b_vram_gate.ipynb`

`configs/derived_budget.yaml` is consumed by `notebooks/01b_vram_gate.ipynb`,
built in parallel by a separate task that will only see this YAML — field
names below match the contract exactly.

In [20]:
import datetime

import yaml

generated_at_utc = datetime.datetime.now(datetime.timezone.utc).isoformat()

derived_budget = {
    "schema_version": 1,
    "generated_at_utc": generated_at_utc,
    "model_revision": MODEL_REVISION,
    "dataset_revision": DATASET_REVISION,
    "processor_size": dict(PROCESSOR_SIZE),
    "patch_size": PATCH_SIZE,
    "spatial_merge_size": SPATIAL_MERGE_SIZE,
    "image_token_ceiling": IMAGE_TOKEN_CEILING,
    "fixed_prompt_and_template_tokens": fixed_prompt_and_template_tokens,
    "eval_prefix_upper_bound": EVAL_PREFIX_UPPER_BOUND,
    "max_new_tokens": MAX_NEW_TOKENS,
    "max_seq_len": MAX_SEQ_LEN,
    "measured_distributions": measured_distributions,
    "schema": {
        "path": "configs/cord_v2_output.schema.json",
        "sha256": schema_hash,
    },
    "lora": {
        "target_modules": r"model\.language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)",
        "r": 16,
        "alpha": 32,
        "dropout": 0.05,
        "expected_trainable_params": 33030144,
    },
}

configs_dir = REPO_ROOT / "configs"
configs_dir.mkdir(parents=True, exist_ok=True)
derived_budget_path = configs_dir / "derived_budget.yaml"
with derived_budget_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(derived_budget, f, sort_keys=False, default_flow_style=False)

print(f"Wrote {derived_budget_path}\n")
print(derived_budget_path.read_text(encoding="utf-8"))

Wrote /Volumes/SSD/Programming/finetuning_vlm/.claude/worktrees/phase2-gate/configs/derived_budget.yaml

schema_version: 1
generated_at_utc: '2026-08-13T14:20:14.539070+00:00'
model_revision: ebb281ec70b05090aa6165b016eac8ec08e71b17
dataset_revision: 7f0115a4b758a71d6473b8d085751692da2fef98
processor_size:
  longest_edge: 1048576
  shortest_edge: 200704
patch_size: 16
spatial_merge_size: 2
image_token_ceiling: 1024
fixed_prompt_and_template_tokens: 321
eval_prefix_upper_bound: 1345
max_new_tokens: 719
max_seq_len: 2064
measured_distributions:
  eval_prefix_len:
    p50: 1329
    p95: 1335
    max: 1345
  train_seq_len:
    p50: 1409
    p95: 1557
    max: 1867
  assistant_label_n:
    p50: 102
    p95: 250
    max: 575
  image_token_n:
    p50: 1008
    p95: 1014
    max: 1024
schema:
  path: configs/cord_v2_output.schema.json
  sha256: 2c69ae8ab48b2d9b7558adba5cdde35780ab6c1034062890c10dc571fba741f9
lora:
  target_modules: model\.language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj

## Part E — Duplication audit (§8.1, §9 deliverable 5)

**Highest-stakes section of this notebook.** This is the only part that
touches the held-out `test` split, and only mechanically: to compute image
hashes and ground-truth hashes for cross-split duplicate detection. Per
`vlm_lab.duplication.public_report()`'s own contract, **only aggregate counts
and the verdict are ever printed** — never hashes, receipt IDs, or pair lists,
because a test-side hash joined against an already-viewed train/validation
image could reveal test content.

Column names are confirmed against the already-loaded `train` split from
Part A (schema-level information only — feature names and types, not row
content), matching the keys `notebooks/01_dataset.ipynb` uses.

In [21]:
print("Column names on the already-loaded train split (schema-level only, no content):")
print(dataset["train"].features)

Column names on the already-loaded train split (schema-level only, no content):
{'image': Image(mode=None, decode=True), 'ground_truth': Value('string')}


In [22]:
import vlm_lab.duplication
import vlm_lab.mechanical_access
from vlm_lab.duplication import ReceiptContent

# Pinned to the same DATASET_REVISION as Part A, per ADR-015 -- the audit must
# run against the exact revision every other measurement in this notebook used,
# not an unpinned "whatever the Hub's default branch currently serves".
all_splits = vlm_lab.mechanical_access.load_all_splits_for_mechanical_check(
    "adr-008-duplication-audit", revision=DATASET_REVISION
)
print("load_all_splits_for_mechanical_check() succeeded (access logged).")

load_all_splits_for_mechanical_check() succeeded (access logged).


In [23]:
receipts_by_split = {
    split_name: [
        ReceiptContent(
            image=row["image"],
            converted_ground_truth=convert_ground_truth(row["ground_truth"]),
        )
        for row in all_splits[split_name]
    ]
    for split_name in ("train", "validation", "test")
}
for split_name, receipts in receipts_by_split.items():
    print(f"{split_name}: {len(receipts)} receipts prepared for hashing (no content displayed).")

train: 800 receipts prepared for hashing (no content displayed).
validation: 100 receipts prepared for hashing (no content displayed).
test: 100 receipts prepared for hashing (no content displayed).


In [24]:
result = vlm_lab.duplication.audit_duplication(receipts_by_split)
report = vlm_lab.duplication.public_report(result)

# PRINT ONLY `report` -- this is the only test-related output this notebook
# may ever print or display.
print(json.dumps(report, indent=2))

{
  "verdict": "EVALUABLE",
  "failed_floors": [],
  "retained_test_receipts": 81,
  "excluded_test_receipts": 19,
  "independent_test_clusters": 81,
  "retained_test_cluster_sizes": [
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1
  ],
  "effective_sample_size": 81.0,
  "retained_validation_receipts": 81,
  "excluded_validation_receipts": 19,
  "excluded_receipts_by_relation": {
    "train-test": 18,
    "validation-test": 2,
    "train-validation": 19
  },
  "floors": {
    

In [25]:
results_dir = REPO_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)
public_report_path = results_dir / "duplication_audit_public.json"
with public_report_path.open("w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(f"Wrote {public_report_path} (this file WILL be committed to git).")

Wrote /Volumes/SSD/Programming/finetuning_vlm/.claude/worktrees/phase2-gate/results/duplication_audit_public.json (this file WILL be committed to git).


In [26]:
# Sealed manifest: receipt_id / split / component_index / relations only --
# NO images, NO ground-truth content. .gitignore already excludes
# results/sealed/, so this is not committed -- but the contents must ALSO
# never be printed to this notebook's own cell output, which WOULD be
# committed (Colab's "Save a copy in GitHub" commits printed cell outputs).
sealed_dir = REPO_ROOT / "results" / "sealed"
sealed_dir.mkdir(parents=True, exist_ok=True)
sealed_manifest_path = sealed_dir / "duplication_audit_manifest.jsonl"
with sealed_manifest_path.open("w", encoding="utf-8") as f:
    for excluded in result.excluded_receipts:
        f.write(json.dumps({
            "receipt_id": excluded.receipt_id,
            "split": excluded.split,
            "component_index": excluded.component_index,
            "relations": list(excluded.relations),
        }, ensure_ascii=False) + "\n")

_sealed_line_count = sum(1 for _ in sealed_manifest_path.open("r", encoding="utf-8"))
print(f"Wrote {_sealed_line_count} line(s) to {sealed_manifest_path} "
      "(sealed; gitignored; contents never printed above or below this line).")

Wrote 38 line(s) to /Volumes/SSD/Programming/finetuning_vlm/.claude/worktrees/phase2-gate/results/sealed/duplication_audit_manifest.jsonl (sealed; gitignored; contents never printed above or below this line).


In [27]:
print(f"Verdict: {result.verdict}")
if result.verdict != vlm_lab.duplication.VERDICT_EVALUABLE:
    print(f"\nNOT EVALUABLE. Failed floor(s): {result.failed_floors}")
    print(
        "\nThis is reported plainly, not softened: per ADR-023, any failed floor means the "
        "confirmatory Phase 5 comparison (or, for the validation floor, Phase 4 checkpoint "
        "selection) cannot proceed as pre-registered without further action."
    )
else:
    print("\nEVALUABLE: all ADR-023 floors (min_retained_receipts, min_independent_clusters, "
          "min_effective_sample_size, min_retained_validation_receipts) passed.")

Verdict: EVALUABLE

EVALUABLE: all ADR-023 floors (min_retained_receipts, min_independent_clusters, min_effective_sample_size, min_retained_validation_receipts) passed.


## Summary

- **§9 deliverable 1 (schema):** `configs/cord_v2_output.schema.json` generated
  from the real train+validation corpus (800 + 100 = 900 converted ground
  truths); round-trip validation confirmed the schema accepts every element of
  its own generating corpus; `schema_hash` computed and printed above.
- **§9 deliverable 2 (token budget):** `eval_prefix_len`, `train_seq_len`,
  `assistant_label_n`, and `image_token_n` measured for real, per-sample, over
  the same 900-sample corpus, using the real pinned processor's own
  `image_grid_thw` (never pixel-area arithmetic). The frozen §4 formulas were
  applied using the real measured `fixed_prompt_and_template_tokens` constant
  and the real `patch_size`/`spatial_merge_size` from the pinned processor
  config, producing real `image_token_ceiling`, `eval_prefix_upper_bound`,
  `max_new_tokens`, and `max_seq_len` values, asserted not to exceed the
  pinned model's real `max_position_embeddings`. See the printed finding above
  regarding §5.4 step 5's literal terminal-EOS check.
- **§9 deliverable 5 (duplication audit):** `vlm_lab.duplication.audit_duplication()`
  run over all three real splits (train=800, validation=100, test=100);
  `public_report()`'s aggregate output printed above and written to
  `results/duplication_audit_public.json`; the sealed per-receipt exclusion
  manifest written to `results/sealed/duplication_audit_manifest.jsonl`
  (gitignored, never printed).
- **Part D:** `configs/derived_budget.yaml` written as the shared artifact
  contract for `notebooks/01b_vram_gate.ipynb`.

### What this notebook explicitly did NOT do

- No model weights were loaded; no `model.generate()` call was made; no
  training occurred.
- `notebooks/02_baseline.ipynb` was not created or executed —
  `docs/proposals/phase1_closure_prereg.md` §9 forbids that until item 9
  (promotion into `EXPERIMENT_SPEC.md` / `EVALUATION_PROTOCOL.md` /
  `IMPLEMENTATION_PLAN.md` / new ADRs) is complete, which is a separate,
  later step this notebook does not perform.
- No test-split image, ground-truth field, or annotation content was
  displayed, printed, or otherwise used to inform any decision in this
  notebook.

### Next steps (outside this notebook's scope)

- `notebooks/01b_vram_gate.ipynb` (a separate task) consumes
  `configs/derived_budget.yaml`.
- §9 item 9 (promoting accepted content into the authoritative spec documents
  and `docs/STATE.md`) remains open.
